# Loading and Cleaning Dataset

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Datasets/IT_Job_Roles_Skills.csv', encoding="ISO-8859-1")
df.head()

,Job Title,Job Description,Skills,Certifications
0,Admin Big Data,Responsible for managing and overseeing big da...,"Hadoop, Spark, MapReduce, Data Lakes, Data War...","Cloudera Certified Professional (CCP), Hortonw..."
1,Ansible Operations Engineer,Focuses on automating IT processes using Ansib...,"Ansible, Linux, Automation, Cloud Platforms, C...",Red Hat Certified Specialist in Ansible Automa...
2,Artifactory Administrator,Manages the Artifactory repository for build a...,"Artifactory, CI/CD, Jenkins, Docker, Maven, Gr...","JFrog Artifactory Certification, DevOps Instit..."
3,Artificial Intelligence / Machine Learning Leader,"Leads AI/ML projects and teams, defining strat...","AI Strategy, Machine Learning, Team Management...","AI-900: Microsoft Azure AI Fundamentals, Certi..."
4,Artificial Intelligence / Machine Learning Sr....,Senior role overseeing multiple AI/ML initiati...,"AI Strategy, Machine Learning, Team Management...",Certified Artificial Intelligence Practitioner...


In [3]:
import re

# Clean the 'Skills' column
df['Skills'] = (
    df['Skills']
    .astype(str)
    .apply(lambda x: re.sub(r"[\[\]'\"']", '', x))  # remove brackets and quotes
    .apply(lambda x: ', '.join([skill.strip() for skill in x.split(',') if skill]))  # clean spaces, keep case
)

# Clean 'Certifications' column
df['Certifications'] = (
    df['Certifications']
    .astype(str)
    .replace({'â€“': '-', 'â€™': "'", 'â€œ': '"', 'â€': '"'}, regex=True)
)

df.head()

,Job Title,Job Description,Skills,Certifications
0,Admin Big Data,Responsible for managing and overseeing big da...,"Hadoop, Spark, MapReduce, Data Lakes, Data War...","Cloudera Certified Professional (CCP), Hortonw..."
1,Ansible Operations Engineer,Focuses on automating IT processes using Ansib...,"Ansible, Linux, Automation, Cloud Platforms, C...",Red Hat Certified Specialist in Ansible Automa...
2,Artifactory Administrator,Manages the Artifactory repository for build a...,"Artifactory, CI/CD, Jenkins, Docker, Maven, Gr...","JFrog Artifactory Certification, DevOps Instit..."
3,Artificial Intelligence / Machine Learning Leader,"Leads AI/ML projects and teams, defining strat...","AI Strategy, Machine Learning, Team Management...","AI-900: Microsoft Azure AI Fundamentals, Certi..."
4,Artificial Intelligence / Machine Learning Sr....,Senior role overseeing multiple AI/ML initiati...,"AI Strategy, Machine Learning, Team Management...",Certified Artificial Intelligence Practitioner...


In [4]:
df['Job Title'].unique()

<ArrowStringArray>
[                                      'Admin Big Data',
                          'Ansible Operations Engineer',
                            'Artifactory Administrator',
    'Artificial Intelligence / Machine Learning Leader',
 'Artificial Intelligence / Machine Learning Sr.Leader',
                    'Artificial intelligence Architect',
                   'Artificial Intelligence Researcher',
                                   'Big Data Architect',
                                    'Big Data Engineer',
                                  'Big Data Specialist',
 ...
                                   'ANIMATION DIRECTOR',
                              'VISUAL EFFECTS ANIMATOR',
                           'GRAPHIC EFFECTS SUPERVISOR',
                                    'FORENSIC ANIMATOR',
                                           'COMPOSITOR',
                   'EFFECTS TECHNICAL DIRECTOR (FX TD)',
                                        'LAYOUT ARTIST',
       

# Enriching Skills Column

In [5]:
# Define role-based skill expansions

role_skills = {
    # --- Data & AI ---
    "data": ["Python", "Pandas", "NumPy", "Matplotlib", "Seaborn", "SQL", "Power BI", "Tableau", "Excel"],
    "machine learning": ["Scikit-learn", "TensorFlow", "Keras", "Python", "Pandas", "NumPy"],
    "big data": ["Hadoop", "Spark", "Kafka", "Hive", "AWS", "ETL", "Scala"],
    "ai": ["Deep Learning", "Neural Networks", "Python", "TensorFlow", "PyTorch"],

    # --- DevOps & Cloud ---
    "devops": ["Docker", "Kubernetes", "Jenkins", "Git", "CI/CD", "AWS", "Terraform"],
    "cloud": ["AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform", "Linux", "Networking"],

    # --- Software Development ---
    "developer": ["Python", "Java", "C++", "Git", "OOP", "REST APIs"],
    "python": ["Python", "Flask", "Django", "Pandas", "NumPy"],
    "java": ["Java", "Spring Boot", "Hibernate", "Maven", "JDBC"],
    "full stack": ["HTML", "CSS", "JavaScript", "React", "Node.js", "MongoDB", "Express"],

    # --- Cybersecurity ---
    "security": ["Network Security", "Firewalls", "Penetration Testing", "Linux", "Ethical Hacking"],
    "cyber": ["Cybersecurity", "Risk Analysis", "IDS/IPS", "SIEM", "Vulnerability Scanning"],

    # --- Network & Infra ---
    "network": ["Networking", "CCNA", "Routers", "Switches", "Firewalls", "Linux"],
    "system": ["Windows Server", "Linux", "Shell Scripting", "Virtualization", "Backup"],

    # --- UI/UX & Design ---
    "ui": ["Figma", "Adobe XD", "HTML", "CSS", "JavaScript", "Wireframing"],
    "ux": ["User Research", "Prototyping", "Usability Testing", "Design Thinking"],
    "designer": ["Photoshop", "Illustrator", "Canva", "Figma"],

    # --- Management ---
    "manager": ["Leadership", "Agile", "Scrum", "Project Planning", "Communication", "Budgeting"],
    "product": ["Market Research", "Roadmaps", "Stakeholder Management", "Agile"],
}

# Function to add extra skills
def enrich_skills(row):
    title = str(row['Job Title']).lower()
    new_skills = set(str(row['Skills']).split(', ')) if pd.notna(row['Skills']) else set()

    for keyword, extra_skills in role_skills.items():
        if keyword in title:
            new_skills.update(extra_skills)

    return ', '.join(sorted(new_skills))

# Apply to dataset
df['Skills'] = df.apply(enrich_skills, axis=1)

df.to_csv("Datasets/job-skill-data.csv", index=False)

# Skills to Jobs Mapping

### Rule-Based Approach

In [6]:
data = pd.read_csv("Datasets/job-skill-data.csv")

In [7]:
# --- Step 1: Get user input ---
user_skills = input("Enter your skills (comma-separated): ").split(',')
user_skills = [skill.strip().lower() for skill in user_skills if skill.strip()]  # lowercase for comparison

# --- Step 2: Function to calculate match ---
def calculate_match(job_skills, user_skills):
    # Convert job skills into lowercase for comparison but keep original
    job_skills_list_original = [s.strip() for s in str(job_skills).split(',') if s.strip()]
    job_skills_list = [s.lower().strip() for s in job_skills_list_original]
    
    matched_skills = set(job_skills_list) & set(user_skills)
    missing_skills = set(job_skills_list) - set(user_skills)
    
    match_percent = (len(matched_skills) / len(job_skills_list)) * 100 if job_skills_list else 0

    # Get missing skills in original form (not lower)
    missing_original = [job_skills_list_original[i] for i, s in enumerate(job_skills_list) if s in missing_skills]
    return round(match_percent, 2), ', '.join(missing_original) if missing_original else 'None'

# --- Step 3: Apply function ---
data[['Match_%', 'Missing_Skills']] = data['Skills'].apply(lambda x: pd.Series(calculate_match(x, user_skills)))

# --- Step 4: Filter top 10 matches ---
matched_jobs = data[data['Match_%'] > 0].sort_values(by='Match_%', ascending=False).head(5)

# Find skill names in their original dataset form (case-matched)
all_skills_original = set()
data['Skills'].apply(lambda x: [all_skills_original.add(s.strip()) for s in str(x).split(',') if s.strip()])

def match_case(skill, all_skills):
    for original in all_skills:
        if original.lower() == skill.lower():
            return original
    return skill.capitalize()

display_user_skills = [match_case(s.strip(), all_skills_original) for s in user_skills if s.strip()]

# --- Step 5: Display results ---
if not matched_jobs.empty:
    print("\n🎯 Top 5 Recommended Jobs Based on Your Skills:\n")
    for _, row in matched_jobs.iterrows():
        print(f"""
Job Title: {row['Job Title']}
Description: {row['Job Description']}
Match: {row['Match_%']}%
Required Skills: {row['Skills']}
Your Skills: {', '.join(display_user_skills)}
Missing Skills: {row['Missing_Skills']}
Certifications: {row['Certifications']}
-----------------------------------------------
""")
else:
    print("\n❌ No matching jobs found.")

Enter your skills (comma-separated):  python



🎯 Top 5 Recommended Jobs Based on Your Skills:


Job Title: ARTIFICIAL INTELLIGENCE RESEARCHER
Description: Conducts research to advance AI technologies, develops new algorithms, and publishes findings.
Match: 25.0%
Required Skills: AI/ML Algorithms, Data Analysis, Python, Research Methodologies
Your Skills: Python
Missing Skills: AI/ML Algorithms, Data Analysis, Research Methodologies
Certifications:  IBM AI Engineering Professional Certificate, Stanford AI Certificate (Online)
-----------------------------------------------


Job Title: ARTIFICIAL INTELLIGENCE ENGINEER
Description: Develops AI models and applications, implementing machine learning algorithms.
Match: 25.0%
Required Skills: AI/ML Algorithms, Deep Learning, Neural Networks, Python
Your Skills: Python
Missing Skills: AI/ML Algorithms, Deep Learning, Neural Networks
Certifications: TensorFlow Developer Certificate, AWS Certified Machine Learning - Specialty
-----------------------------------------------


Job Title: NAT

### Using Machine Learning (NLP)

In [8]:
# --- Import Libraries ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Step 1: Preprocess Data ---
data['Skills'] = data['Skills'].fillna('').astype(str)
data['Job Title'] = data['Job Title'].fillna('').astype(str)

# --- Step 2: Create TF-IDF Vectors ---
skill_vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r'[^,]+')

skill_vectors = skill_vectorizer.fit_transform(data['Skills'])

# --- Step 3: Function to Recommend Jobs ---
def recommend_jobs(user_input, top_n=10):
    # Clean user input
    user_input_clean = ', '.join([s.strip().lower() for s in user_input.split(',') if s.strip()])
    user_vector = skill_vectorizer.transform([user_input_clean])

    # Cosine similarity with job skill vectors
    similarity = cosine_similarity(user_vector, skill_vectors).flatten()
    result = data.copy()
    result['Match_%'] = (similarity * 100).round(2)

    # Calculate missing skills for each job
    def find_missing(job_skills, user_skills):
        job_skills_list = [s.strip() for s in str(job_skills).split(',') if s.strip()]
        missing = [s for s in job_skills_list if s.lower() not in user_skills]
        return ', '.join(missing) if missing else 'None'

    user_skills_list = [s.strip().lower() for s in user_input.split(',') if s.strip()]
    data['Missing_Skills'] = data['Skills'].apply(lambda x: find_missing(x, user_skills_list))

    # Return top N jobs
    return data.sort_values(by='Match_%', ascending=False).head(top_n)

# --- Step 4: Input from User ---
user_skills = input("Enter your skills (comma-separated): ")
top_n_jobs = int(input("Top n jobs to be listed: "))

# --- Step 5: Get Top Matches ---
matched_jobs = recommend_jobs(user_skills, top_n_jobs)

# --- Step 6: Display Skills in Original Case (for clean output) ---
all_skills_original = set()
data['Skills'].apply(lambda x: [all_skills_original.add(s.strip()) for s in str(x).split(',') if s.strip()])

def match_case(skill, all_skills):
    for original in all_skills:
        if original.lower() == skill.lower():
            return original
    return skill.capitalize()

display_user_skills = [match_case(s.strip(), all_skills_original) for s in user_skills.split(',') if s.strip()]

# --- Step 7: Display Final Results ---
if not matched_jobs.empty:
    print(f"\n🎯 Top {top_n_jobs} Recommended Jobs Based on Your Skills:\n")
    for _, row in matched_jobs.iterrows():
        print(f"""
Job Title: {row['Job Title']}
Description: {row['Job Description']}
Match: {row['Match_%']}%
Required Skills: {row['Skills']}
Your Skills: {', '.join(display_user_skills)}
Missing Skills: {row['Missing_Skills']}
Certifications: {row['Certifications']}
-----------------------------------------------
""")
else:
    print("\n❌ No matching jobs found.")

Enter your skills (comma-separated):  python
Top n jobs to be listed:  3



🎯 Top 3 Recommended Jobs Based on Your Skills:


Job Title: ARTIFICIAL INTELLIGENCE RESEARCHER
Description: Conducts research to advance AI technologies, develops new algorithms, and publishes findings.
Match: 25.0%
Required Skills: AI/ML Algorithms, Data Analysis, Python, Research Methodologies
Your Skills: Python
Missing Skills: AI/ML Algorithms, Data Analysis, Research Methodologies
Certifications:  IBM AI Engineering Professional Certificate, Stanford AI Certificate (Online)
-----------------------------------------------


Job Title: NATURAL LANGUAGE PROCESSING ENGINEER
Description: Develops NLP models for various applications, such as chatbots and text analysis tools.
Match: 25.0%
Required Skills: Machine Learning, NLP Algorithms, Python, Text Analysis
Your Skills: Python
Missing Skills: Machine Learning, NLP Algorithms, Text Analysis
Certifications: Certified Mobile Device Security Professional (CMDSP), GIAC Mobile Device Security Analyst (GMOB)
--------------------------------

### Job to Skills Mapping

In [9]:
job_title_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english'
)

job_title_vectors = job_title_vectorizer.fit_transform(
    data['Job Title'].fillna('').astype(str)
)

def job_to_skills_mapping(job_title, top_n=5):

    job_vector = job_title_vectorizer.transform([job_title.lower()])
    similarity = cosine_similarity(job_vector, job_title_vectors).flatten()

    temp = data.copy()
    temp['Match_%'] = (similarity * 100).round(2)

    top_jobs = temp.sort_values(by='Match_%', ascending=False).head(top_n)

    # Collect skills from matched jobs
    skills_set = set()
    for skills in top_jobs['Skills']:
        for skill in skills.split(','):
            skills_set.add(skill.strip())

    return {
        "Job Title": job_title,
        "Recommended Skills": sorted(skills_set),
        "Top Matching Jobs": top_jobs[['Job Title', 'Match_%']]
    }

job_title_input = input("Enter a job title: ")
top_n_jobs = int(input("Enter number of jobs: "))

job_to_skills_mapping(job_title_input, top_n_jobs)

Enter a job title:  data analyst
Enter number of jobs:  2


{'Job Title': 'data analyst',
 'Recommended Skills': ['Data Analysis',
  'Data Cleansing',
  'Data Mining',
  'Data Modeling',
  'Data Visualization',
  'ETL',
  'Excel',
  'Matplotlib',
  'NumPy',
  'Pandas',
  'Power BI',
  'Python',
  'R',
  'SQL',
  'Seaborn',
  'Statistical Analysis',
  'Statistics',
  'Tableau'],
 'Top Matching Jobs':         Job Title  Match_%
 247  DATA ANALYST    100.0
 70   DATA ANALYST    100.0}

In [10]:
import pickle

# After fitting TF-IDF
with open('models/skill_vectorizer.pkl', 'wb') as f:
    pickle.dump(skill_vectorizer, f)

with open('models/skill_vectors.pkl', 'wb') as f:
    pickle.dump(skill_vectors, f)

with open('models/job_title_vectorizer.pkl', 'wb') as f:
    pickle.dump(job_title_vectorizer, f)

with open('models/job_title_vectors.pkl', 'wb') as f:
    pickle.dump(job_title_vectors, f)